In [2]:
import os
from pathlib import Path
import pandas as pd

print(os.getcwd())
DATA = Path.home() / "churn-project" / "data" / "telco_churn.csv"
df = pd.read_csv(DATA)
print(df.shape)
df.info()
df["Churn"].value_counts(normalize=True)


C:\Users\aniru\churn-project
(7043, 21)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBill

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
blank = df[df["TotalCharges"].isna()]
print(len(blank), "rows had a blank TotalCharges")
print(blank[["customerID", "tenure", "MonthlyCharges"]])

11 rows had a blank TotalCharges
      customerID  tenure  MonthlyCharges
488   4472-LVYGI       0           52.55
753   3115-CZMZD       0           20.25
936   5709-LVOEQ       0           80.85
1082  4367-NUYAO       0           25.75
1340  1371-DWPAZ       0           56.05
3331  7644-OMVMY       0           19.85
3826  3213-VVOLG       0           25.35
4380  2520-SGTTA       0           20.00
5218  2923-ARZLG       0           19.70
6670  4075-WKNIU       0           73.35
6754  2775-SEFEE       0           61.90


In [4]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["Churn"] = (df["Churn"] == "Yes").astype(int)
print(df["TotalCharges"].dtype, df["TotalCharges"].isna().sum())
print(df["Churn"].value_counts())

float64 0
Churn
0    5174
1    1869
Name: count, dtype: int64


In [5]:
for col in ["Contract", "InternetService", "PaymentMethod", "TechSupport"]:
    print(df.groupby(col)["Churn"].mean().sort_values(ascending=False).round(3), "\n")

df["tenure_band"] = pd.cut(df["tenure"], [-1, 6, 12, 24, 48, 72],
                           labels=["0-6m", "7-12m", "1-2y", "2-4y", "4-6y"])
print(df.groupby("tenure_band", observed=True)["Churn"].mean().round(3))

Contract
Month-to-month    0.427
One year          0.113
Two year          0.028
Name: Churn, dtype: float64 

InternetService
Fiber optic    0.419
DSL            0.190
No             0.074
Name: Churn, dtype: float64 

PaymentMethod
Electronic check             0.453
Mailed check                 0.191
Bank transfer (automatic)    0.167
Credit card (automatic)      0.152
Name: Churn, dtype: float64 

TechSupport
No                     0.416
Yes                    0.152
No internet service    0.074
Name: Churn, dtype: float64 

tenure_band
0-6m     0.529
7-12m    0.359
1-2y     0.287
2-4y     0.204
4-6y     0.095
Name: Churn, dtype: float64


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

X = df.drop(columns=["customerID", "Churn", "tenure_band"])
y = df["Churn"]

num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat_cols = [c for c in X.columns if c not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])),
    ("clf", LogisticRegression(max_iter=1000))])

model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, pred, digits=3))
print("ROC AUC:", round(roc_auc_score(y_test, proba), 3))

              precision    recall  f1-score   support

           0      0.849     0.895     0.871      1035
           1      0.657     0.559     0.604       374

    accuracy                          0.806      1409
   macro avg      0.753     0.727     0.738      1409
weighted avg      0.798     0.806     0.800      1409

ROC AUC: 0.842


In [8]:
model.set_params(clf__class_weight="balanced")
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, pred, digits=3))
print("ROC AUC:", round(roc_auc_score(y_test, proba), 3))

              precision    recall  f1-score   support

           0      0.902     0.722     0.802      1035
           1      0.504     0.783     0.614       374

    accuracy                          0.738      1409
   macro avg      0.703     0.753     0.708      1409
weighted avg      0.797     0.738     0.752      1409

ROC AUC: 0.842


In [9]:
names = model.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(model.named_steps["clf"].coef_[0], index=names).sort_values()
print("Keeps customers:\n", coefs.head(8).round(2))
print("\nDrives churn:\n", coefs.tail(8).round(2))

Keeps customers:
 num__tenure                                 -1.15
cat__Contract_Two year                      -0.78
num__MonthlyCharges                         -0.68
cat__InternetService_DSL                    -0.62
cat__DeviceProtection_No internet service   -0.28
cat__OnlineBackup_No internet service       -0.28
cat__OnlineSecurity_No internet service     -0.28
cat__TechSupport_No internet service        -0.28
dtype: float64

Drives churn:
 cat__TechSupport_No                    0.17
cat__OnlineSecurity_No                 0.20
cat__PaymentMethod_Electronic check    0.24
cat__StreamingTV_Yes                   0.26
cat__StreamingMovies_Yes               0.28
num__TotalCharges                      0.49
cat__Contract_Month-to-month           0.66
cat__InternetService_Fiber optic       0.71
dtype: float64


In [10]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = Pipeline([
    ("prep", ColumnTransformer([
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])),
    ("clf", HistGradientBoostingClassifier(class_weight="balanced", max_depth=3,
                                           learning_rate=0.05, max_iter=300, random_state=42))])
gb.fit(X_train, y_train)
pred = gb.predict(X_test)
print(classification_report(y_test, pred, digits=3))
print("ROC AUC:", round(roc_auc_score(y_test, gb.predict_proba(X_test)[:, 1]), 3))

              precision    recall  f1-score   support

           0      0.908     0.727     0.807      1035
           1      0.513     0.797     0.624       374

    accuracy                          0.745      1409
   macro avg      0.711     0.762     0.716      1409
weighted avg      0.803     0.745     0.759      1409

ROC AUC: 0.841


In [11]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, m in [("LogReg", model), ("GradBoost", gb)]:
    s = cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name}: {s.mean():.3f} ± {s.std():.3f}")
    

LogReg: 0.846 ± 0.012
GradBoost: 0.845 ± 0.011


In [12]:
import joblib
(Path.home() / "churn-project" / "models").mkdir(exist_ok=True)
joblib.dump(model, Path.home() / "churn-project" / "models" / "logreg_churn.joblib")
print("saved")

saved
